# Synthetic Dataset Playground (Parameter-Controlled)

This notebook is a **parameter-first** version of your `gen_syth_data.py` generator.

Key idea: **every stochastic choice is parameterized**. For any parameter you can supply either:
- a **fixed scalar** (e.g., `0.05`) to make it deterministic, or
- a **range tuple** (e.g., `(0.03, 0.08)`) to sample uniformly at runtime.

You can therefore “turn off randomness” gradually and study how each component affects:
- sample appearance,
- anomaly localization,
- runtime / detection performance downstream.


In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Union, Optional

import numpy as np
import matplotlib.pyplot as plt

FloatOrRange = Union[float, Tuple[float, float]]
IntOrRange   = Union[int,   Tuple[int, int]]

def sample_uniform(rng: np.random.Generator, v: FloatOrRange) -> float:
    if isinstance(v, tuple):
        return float(rng.uniform(v[0], v[1]))
    return float(v)

def sample_int(rng: np.random.Generator, v: IntOrRange) -> int:
    if isinstance(v, tuple):
        lo, hi = int(v[0]), int(v[1])
        if hi < lo:
            lo, hi = hi, lo
        return int(rng.integers(lo, hi + 1))
    return int(v)

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

## 1) Dataset Group Specification

Each group defines:
- `length`: number of channels per spectrum
- `W`: typical scan window cap (used only to set generator scale)
- `n_rows`: number of rows in the group


In [2]:
@dataclass
class SpectrumSpec:
    length: int
    W: int
    n_rows: int

DEFAULT_GROUPS: List[SpectrumSpec] = [
    SpectrumSpec(100,  5, 50),
    SpectrumSpec(100, 10, 50),
    SpectrumSpec(200, 10, 50),
    SpectrumSpec(200, 20, 50),
    SpectrumSpec(500, 25, 50),
    SpectrumSpec(500, 50, 50),
]

## 2) Top-level knobs

These control *probabilities* and *global behavior*. Everything else is in component parameter dictionaries below.


In [21]:
# Reproducibility
SEED = 123

# Event probabilities
P_STRONG_ANOM = 0.05
P_WEAK_LINE   = 0.25
P_INTERFERENCE= 0

# Ignore buffers near band edges (fraction of length)
BUFFER_FRAC = 0

# Output directory for plots (optional)
OUT_ROOT = "Images/SyntheticPlots"


## 3) Parameter dictionaries (editable)

All parameter ranges below correspond to your current code defaults, but now you can:
- fix any parameter by replacing `(a,b)` with a scalar
- widen/narrow ranges to change variability


In [34]:
# ------------------------------
# Bandpass-like envelope parameters
# ------------------------------
ENV_PARAMS = {
    "left_scale":  (0.03, 0.08),
    "right_scale": (0.03, 0.08),
    "floor":       (0.90, 0.96),
}

# ------------------------------
# Trend parameters
# ------------------------------
TREND_PARAMS = {
    "level": (0.97, 1.01),
    "slope": (-0.03, 0.05),
    "quad":  (-0.03, 0.03),
}

# ------------------------------
# Low-frequency structure parameters
# ------------------------------
FIELD_PARAMS = {
    # sigma_bins is derived from W by default; you can override by setting a scalar.
    # if None -> sigma_bins = max(6.0, 0.9*W)
    "sigma_bins_override": None,   # e.g. 12.0
    "field_amp": (0.002, 0.010),   # multiplicative amplitude

    # extra weak sinusoids
    "n_sinusoids": (1, 2),         # integers
    "k_cycles":    (0.5, 2.5),
    "amp":         (0.0005, 0.0030),
}

# ------------------------------
# Noise parameters
# ------------------------------
NOISE_PARAMS = {
    # Relative to median(y)
    "white_sigma_frac": (0.0015, 0.0060),
    "corr_sigma_frac":  (0.0006, 0.0030),

    # AR(1) correlation
    "rho":              (0.70, 0.93),

    # mixture weight: sqrt(a)*white + sqrt(1-a)*correlated
    "white_frac":       (0.70, 0.92),

    # edge heteroskedasticity
    "edge_beta":        (0.2, 0.8),
}

# ------------------------------
# Strong anomaly controls (3 types)
# ------------------------------

# Probability a row contains a strong anomaly (ground-truth positive)
P_STRONG_ANOM = 0.05

# Number of strong features per positive row (inclusive range)
STRONG_NUM_FEATURES = (1, 1)

# Mixture over strong feature *types* when strong_kind="mixture"
# (must sum to 1.0)
STRONG_TYPE_PROBS = {
    "gauss_abs": 0.60,
    "step":      0.30,
    "gauss_em":  0.10,
}

# Gaussian absorption / emission parameters (match your LaTeX narrative)
STRONG_GAUSS_ABS_PARAMS = {
    "center_range": (0.10, 0.90),      # u ~ Unif(0.1, 0.9)
    "width_frac_range": (0.01, 0.05),  # omega fraction of full domain
    "amp_range": (0.70, 1.20),         # A range (multiplies local level)
    "label_thresh": 0.05,              # record indices where g > 0.05
}

STRONG_GAUSS_EM_PARAMS = {
    "center_range": (0.10, 0.90),
    "width_frac_range": (0.005, 0.03),
    "amp_range": (0.50, 1.00),
    "label_thresh": 0.05,
}

# Rectangular step parameters
STRONG_STEP_PARAMS = {
    "width": 0.01,  # m is a fraction of n
    "strength": 0.05,       # S in Delta = S * local_level
}

# ------------------------------
# Weak lines parameters
# ------------------------------
WEAK_LINE_PARAMS = {
    "n_lines":         (1, 2),         # integers
    "absorption_prob": 0.75,
    "amp_frac":        (0.003, 0.012),
    "width_frac":      (0.003, 0.010),
    "center_frac":     (0.15, 0.85),   # where line center can appear
}

# ------------------------------
# Interference spike parameters (ignored regions)
# ------------------------------
INTERFERENCE_PARAMS = {
    "amp_mult":   (3.0, 7.0),
    "left_region":(0.06, 0.18),        # spike location fraction of n
    "halfwidth":  2,                   # ignore interval half-width in bins
}


## 4) Generator implementation (parameter-controlled)

This is the same structure you had, but with **all randomness routed through the parameter dictionaries**.


In [14]:
# ------------------------------
# Helpers: smooth continuum
# ------------------------------

def band_envelope(n: int, rng: np.random.Generator, params: Dict[str, Any]) -> np.ndarray:
    x = np.linspace(0.0, 1.0, n)

    left_scale  = sample_uniform(rng, params["left_scale"])
    right_scale = sample_uniform(rng, params["right_scale"])

    left_rise  = 1.0 - np.exp(-x / max(left_scale, 1e-12))
    right_fall = 1.0 - np.exp(-(1.0 - x) / max(right_scale, 1e-12))

    env = left_rise * right_fall
    env /= (env.max() + 1e-12)

    floor = sample_uniform(rng, params["floor"])
    return floor + (1.0 - floor) * env

def lowpass_random_field(n: int, rng: np.random.Generator, sigma_bins: float) -> np.ndarray:
    z = rng.normal(0.0, 1.0, size=n).astype(np.float64)

    r = int(max(3, round(4.0 * sigma_bins)))
    t = np.arange(-r, r + 1, dtype=np.float64)
    k = np.exp(-0.5 * (t / max(sigma_bins, 1e-6)) ** 2)
    k /= (k.sum() + 1e-12)

    y = np.convolve(z, k, mode="same")
    y -= y.mean()
    y /= (y.std() + 1e-12)
    return y

def continuum_like(n: int, W: int, rng: np.random.Generator,
                   env_params: Dict[str, Any],
                   trend_params: Dict[str, Any],
                   field_params: Dict[str, Any]) -> np.ndarray:
    x = np.linspace(0.0, 1.0, n)
    # env = band_envelope(n, rng, env_params)

    level = sample_uniform(rng, trend_params["level"])
    slope = sample_uniform(rng, trend_params["slope"])
    quad  = sample_uniform(rng, trend_params["quad"])
    trend = 1.0 + slope * (x - 0.5) + quad * (x - 0.5) ** 2

    if field_params.get("sigma_bins_override") is None:
        sigma_bins = max(6.0, 0.9 * float(W))
    else:
        sigma_bins = float(field_params["sigma_bins_override"])

    field = lowpass_random_field(n, rng, sigma_bins=sigma_bins)
    field_amp = sample_uniform(rng, field_params["field_amp"])

    y = level * trend * (1.0 + field_amp * field)

    n_sin = sample_int(rng, field_params["n_sinusoids"])
    for _ in range(n_sin):
        k = sample_uniform(rng, field_params["k_cycles"])
        phase = sample_uniform(rng, (0.0, 2*np.pi))
        amp = sample_uniform(rng, field_params["amp"])
        y += amp * np.sin(2*np.pi*k*x + phase)

    return y.astype(np.float64)

# ------------------------------
# Helpers: noise (white + correlated) with edge profile
# ------------------------------

def edge_profile(n: int) -> np.ndarray:
    x = np.linspace(0.0, 1.0, n)
    d = np.minimum(x, 1.0 - x)
    prof = 1.0 - (d / (0.5 + 1e-12))
    return prof**2

def add_noise(y: np.ndarray, rng: np.random.Generator, noise_params: Dict[str, Any]) -> None:
    n = y.size
    level = float(np.median(y))
    level = max(level, 1e-6)

    sigma_w0 = level * sample_uniform(rng, noise_params["white_sigma_frac"])
    sigma_c0 = level * sample_uniform(rng, noise_params["corr_sigma_frac"])
    rho      = sample_uniform(rng, noise_params["rho"])
    a        = sample_uniform(rng, noise_params["white_frac"])
    edge_beta= sample_uniform(rng, noise_params["edge_beta"])

    g = edge_profile(n)
    sigma_w = sigma_w0 * (1.0 + edge_beta * g)
    sigma_c = sigma_c0 * (1.0 + edge_beta * g)

    eps = rng.normal(0.0, 1.0, size=n)

    xi = np.empty(n, dtype=np.float64)
    xi[0] = rng.normal(0.0, 1.0)
    s = np.sqrt(max(1.0 - rho*rho, 1e-12))
    for i in range(1, n):
        xi[i] = rho * xi[i-1] + s * rng.normal(0.0, 1.0)

    y += np.sqrt(a) * (sigma_w * eps) + np.sqrt(1.0 - a) * (sigma_c * xi)

def _normalize_probs(prob_dict: dict) -> tuple[list[str], np.ndarray]:
    keys = list(prob_dict.keys())
    p = np.array([float(prob_dict[k]) for k in keys], dtype=np.float64)
    s = float(p.sum())
    if s <= 0:
        raise ValueError("STRONG_TYPE_PROBS must have positive mass.")
    return keys, (p / s)

def _add_strong_gaussian_feature(
    y: np.ndarray,
    rng: np.random.Generator,
    kind: str,  # "absorption" or "emission"
    params: dict,
) -> tuple[int, int]:
    n = y.size
    ii = np.arange(n, dtype=np.float64)

    u = rng.uniform(*params["center_range"])
    omega = rng.uniform(*params["width_frac_range"])  # fraction of domain

    # Convert width fraction to sigma in bins using FWHM ≈ 2.355 sigma
    sigma = max(1e-6, (omega * (n - 1)) / 2.355)
    cc = u * (n - 1)

    g = np.exp(-0.5 * ((ii - cc) / sigma) ** 2)  # peak = 1

    # Local reference level (robust): median in a small neighborhood
    c0 = int(round(cc))
    lo = max(0, c0 - 30)
    hi = min(n, c0 + 31)
    ell = float(np.median(y[lo:hi])) if hi > lo else float(np.median(y))
    ell = max(ell, 1e-12)

    A = rng.uniform(*params["amp_range"])
    amp = A * ell

    if kind == "absorption":
        y -= amp * g
        # minimal non-negativity constraint
        y[:] = np.maximum(y, 0.0)
    elif kind == "emission":
        y += amp * g
    else:
        raise ValueError(f"Unknown Gaussian kind={kind!r}")

    # Label support as contiguous region where g > threshold (threshold is on peak-normalized g)
    thr = float(params.get("label_thresh", 0.05))
    idx = np.where(g > thr)[0]
    if idx.size == 0:
        return c0, c0
    return int(idx[0]), int(idx[-1])

def _add_rectangular_step(
    y: np.ndarray,
    rng: np.random.Generator,
    params: dict,
) -> tuple[int, int]:
    n = y.size
    width = int(params['width'] * n) 
    strength = params['strength']
    start = int(rng.integers(int(0.1 * n), int(0.9 * n - width)))
    end = start + width - 1

    seg = y[start:end + 1].copy()                 # keep existing noisy segment
    local_level = float(seg.mean())
    offset = strength * local_level

    if rng.random() < 0.5:
        y[start:end + 1] = seg + offset           # up-step, noise preserved
    else:
        y[start:end + 1] = np.maximum(0.0, seg - offset)  # down-step, noise preserved

    return start, end


def _inject_strong_anomalies(
    y: np.ndarray,
    rng: np.random.Generator,
    strong_kind: str = "mixture",  # {"mixture","gauss_abs","step","gauss_em"}
) -> list[tuple[int, int]]:
    intervals: list[tuple[int, int]] = []

    F_lo, F_hi = int(STRONG_NUM_FEATURES[0]), int(STRONG_NUM_FEATURES[1])
    # F = int(rng.integers(F_lo, F_hi + 1))
    F = 1

    keys, p = _normalize_probs(STRONG_TYPE_PROBS)

    for _ in range(F):
        if strong_kind == "mixture":
            t = str(rng.choice(np.array(keys, dtype=object), p=p))
        else:
            t = strong_kind

        if t == "gauss_abs":
            s, e = _add_strong_gaussian_feature(
                y, rng, kind="absorption", params=STRONG_GAUSS_ABS_PARAMS
            )
        elif t == "gauss_em":
            s, e = _add_strong_gaussian_feature(
                y, rng, kind="emission", params=STRONG_GAUSS_EM_PARAMS
            )
        elif t == "step":
            s, e = _add_rectangular_step(y, rng, params=STRONG_STEP_PARAMS)
        else:
            raise ValueError(f"Unknown strong anomaly type {t!r}")

        intervals.append((int(s), int(e)))

    return intervals

# ------------------------------
# Weak lines (subtle)
# ------------------------------

def add_weak_line(y: np.ndarray, rng: np.random.Generator, weak_params: Dict[str, Any], kind: str) -> Tuple[int, int]:
    n = y.size
    ii = np.arange(n, dtype=np.float64)

    center = sample_uniform(rng, weak_params["center_frac"])
    width_frac = sample_uniform(rng, weak_params["width_frac"])

    sigma = max(1e-6, (width_frac / 2.355) * n)
    cc = center * (n - 1)

    prof = np.exp(-0.5 * ((ii - cc) / sigma) ** 2)

    local = float(np.median(y[max(0, int(cc-30)):min(n, int(cc+30))]))
    amp = sample_uniform(rng, weak_params["amp_frac"]) * max(local, 1e-6)

    if kind == "absorption":
        y -= amp * prof
    else:
        y += amp * prof

    mask = prof > 0.2
    idx = np.where(mask)[0]
    if idx.size == 0:
        return int(cc), int(cc)
    return int(idx[0]), int(idx[-1])

# ------------------------------
# Interference spikes (ignored)
# ------------------------------

def add_interference_spike(y: np.ndarray, rng: np.random.Generator, params: Dict[str, Any]) -> Tuple[int, int]:
    n = y.size
    k = int(sample_uniform(rng, params["left_region"]) * n)
    k = int(np.clip(k, 0, n-1))

    local = float(np.median(y[max(0, k-20):min(n, k+20)]))
    spike = sample_uniform(rng, params["amp_mult"]) * max(local, 1e-6)

    y[k] += spike

    hw = int(params.get("halfwidth", 2))
    s = max(0, k-hw)
    e = min(n-1, k+hw)
    return s, e


## 5) Single-spectrum API

In [22]:
def generate_single_spectrum(
    length: int,
    W: int,
    rng: np.random.Generator,
    p_strong_anom: float = P_STRONG_ANOM,
    p_weak_line: float = P_WEAK_LINE,
    # p_interference: float = P_INTERFERENCE,
    buffer_frac: float = BUFFER_FRAC,
    env_params: Dict[str, Any] = ENV_PARAMS,
    trend_params: Dict[str, Any] = TREND_PARAMS,
    field_params: Dict[str, Any] = FIELD_PARAMS,
    noise_params: Dict[str, Any] = NOISE_PARAMS,
    weak_params: Dict[str, Any] = WEAK_LINE_PARAMS,
    # interference_params: Dict[str, Any] = INTERFERENCE_PARAMS,
    strong_kind: str = "mixture",
) -> Dict[str, Any]:
    y = continuum_like(length, W, rng, env_params, trend_params, field_params)
    add_noise(y, rng, noise_params)

    strong_intervals: List[Tuple[int, int]] = []
    weak_intervals: List[Tuple[int, int]] = []
    ignore_ranges: List[Tuple[int, int]] = []

    buf = int(round(buffer_frac * length))
    if buf > 0:
        ignore_ranges.append((0, buf-1))
        ignore_ranges.append((length-buf, length-1))

    has_strong = (rng.random() < p_strong_anom)
    if has_strong:
        strong_intervals.extend(_inject_strong_anomalies(y, rng, strong_kind=strong_kind))

    if rng.random() < p_weak_line:
        n_lines = sample_int(rng, weak_params["n_lines"])
        for _ in range(n_lines):
            kind = "absorption" if rng.random() < float(weak_params.get("absorption_prob", 0.75)) else "emission"
            s, e = add_weak_line(y, rng, weak_params, kind=kind)
            # weak_intervals.append((s, e))

    # if (not has_strong) and (rng.random() < p_interference):
    #     s_i, e_i = add_interference_spike(y, rng, interference_params)
    #     ignore_ranges.append((s_i, e_i))

    return {
        "y": y.astype(np.float32),
        "has_strong_anom": bool(has_strong),
        "strong_intervals": strong_intervals,
        "weak_intervals": weak_intervals,
    }

## 6) Dataset generator (API-compatible)

In [23]:
def generate_synthetic_dataset(
    groups: Optional[List[SpectrumSpec]] = None,
    seed: int = SEED,
    strong_rate: float = P_STRONG_ANOM,
    strong_kind: str = "mixture",
) -> Dict[str, Any]:
    if groups is None:
        groups = DEFAULT_GROUPS
        

    rng = np.random.default_rng(seed)
    all_groups: List[Dict[str, Any]] = []

    for gid, spec in enumerate(groups):
        length = int(spec.length)
        n_rows = int(spec.n_rows)

        spectra = np.empty((n_rows, length), dtype=np.float32)
        strong_flags = np.zeros(n_rows, dtype=bool)

        strong_labels: List[List[Tuple[int, int]]] = []
        weak_labels: List[List[Tuple[int, int]]] = []
        # ignore_labels: List[List[Tuple[int, int]]] = []

        # --- stage 0: decide EXACT strong indices for this group ---
        k = int(np.round(strong_rate * n_rows))   # exact count per group
        k = int(np.clip(k, 0, n_rows))
        strong_rows = np.zeros(n_rows, dtype=bool)
        if k > 0:
            chosen = rng.choice(np.arange(n_rows), size=k, replace=False)
            strong_rows[chosen] = True

        for r in range(n_rows):
            # Always generate a baseline spectrum first (NO strong anomaly)
            row = generate_single_spectrum(
                length=length,
                W=spec.W,
                rng=rng,
                p_strong_anom=0.0,         # force baseline
                p_weak_line=0.25,
                # p_interference=0.08,
                buffer_frac=0.03,
                strong_kind=strong_kind,
            )

            y = row["y"].astype(np.float64)  # work in float64 for edits

            strong_intervals = list(row["strong_intervals"])  # should be empty now
            weak_intervals = list(row["weak_intervals"])
            # ignore_ranges = list(row["ignore_ranges"])

            # --- stage 1: inject strong anomaly if this row was selected ---
            if strong_rows[r]:
                strong_intervals = _inject_strong_anomalies(y, rng, strong_kind=strong_kind)
                has_strong = True
            else:
                has_strong = False

            spectra[r, :] = y.astype(np.float32)
            strong_flags[r] = has_strong
            strong_labels.append(strong_intervals)
            weak_labels.append(weak_intervals)
            # ignore_labels.append(ignore_ranges)

        all_groups.append({
            "group_id": int(gid),
            "length": length,
            "W": int(spec.W),
            "spectra": spectra,
            "has_strong_anom": strong_flags,
            "strong_labels": strong_labels,
            "weak_labels": weak_labels,
            # "ignore_ranges": ignore_labels,
        })

    return {"groups": all_groups}

## 7) Visualization helpers

In [24]:
def plot_single_example(length: int = 200, W: int = 20, seed: int = 0) -> None:
    rng = np.random.default_rng(seed)
    row = generate_single_spectrum(length=length, W=W, rng=rng)

    y = row["y"].astype(np.float64)
    x = np.arange(length)

    plt.figure(figsize=(10, 3))
    plt.plot(x, y, linewidth=1.0)
    plt.title(f"Single synthetic spectrum (L={length}, W={W}) | strong={row['has_strong_anom']}")
    plt.xlabel("Index")
    plt.ylabel("Amplitude")

    for (s,e) in row["ignore_ranges"]:
        plt.axvspan(s, e, alpha=0.15, color="0.5")
    for (s,e) in row["weak_intervals"]:
        plt.axvspan(s, e, alpha=0.20, color="C2")
    for (s,e) in row["strong_intervals"]:
        plt.axvspan(s, e, alpha=0.25, color="C1")

    plt.tight_layout()
    plt.show()

def plot_group_samples(dataset: Dict[str, Any],
                       n_no_strong: int = 3,
                       n_with_strong: int = 3,
                       seed: int = 0,
                       out_root: Optional[str] = None) -> None:
    rng = np.random.default_rng(seed)
    if out_root is not None:
        ensure_dir(out_root)

    for group in dataset["groups"]:
        L = int(group["length"])
        W = int(group["W"])
        X = group["spectra"]
        has_strong = group["has_strong_anom"]
        strong_labels = group["strong_labels"]
        weak_labels = group["weak_labels"]
        # ignore_ranges = group["ignore_ranges"]

        idx_with = np.where(has_strong)[0]
        idx_without = np.where(~has_strong)[0]

        # if idx_with.size == 0 or idx_without.size == 0:
        #     continue

        if idx_with.size > n_with_strong:
            idx_with = rng.choice(idx_with, size=n_with_strong, replace=False)
        if idx_without.size > n_no_strong:
            idx_without = rng.choice(idx_without, size=n_no_strong, replace=False)

        rows = max(idx_with.size, idx_without.size)
        fig, axes = plt.subplots(rows, 2, figsize=(12, 2.5*rows), squeeze=False)

        def shade(ax, ranges, color, alpha):
            for (s,e) in ranges:
                ax.axvspan(s, e, color=color, alpha=alpha)

        for r in range(rows):
            ax = axes[r, 0]
            if r < idx_without.size:
                ridx = int(idx_without[r])
                ax.plot(X[ridx], linewidth=1.0)
                # shade(ax, ignore_ranges[ridx], "0.5", 0.15)
                shade(ax, weak_labels[ridx], "C2", 0.20)
                ax.set_title(f"L={L}, W={W} | row={ridx} (no strong)")
            else:
                ax.axis("off")

            ax = axes[r, 1]
            if r < idx_with.size:
                ridx = int(idx_with[r])
                ax.plot(X[ridx], linewidth=1.0)
                # shade(ax, ignore_ranges[ridx], "0.5", 0.15)
                shade(ax, weak_labels[ridx], "C2", 0.20)
                shade(ax, strong_labels[ridx], "C1", 0.25)
                ax.set_title(f"L={L}, W={W} | row={ridx} (strong)")
            else:
                ax.axis("off")

        fig.suptitle(f"Group samples (L={L}, W={W})", y=0.99)
        fig.tight_layout(rect=[0,0,1,0.97])

        if out_root is not None:
            out_path = os.path.join(out_root, f"synthetic_L{L}_W{W}_samples.png")
            fig.savefig(out_path, dpi=150)
            print(f"Saved {out_path}")
            plt.close(fig)
        else:
            plt.show()

## 8) Try it: generate and inspect

In [25]:
# # Quick sanity check: single example
# plot_single_example(length=200, W=20, seed=1)

# # Generate dataset and plot a few samples per group
# data = generate_synthetic_dataset(seed=SEED, strong_rate=P_STRONG_ANOM)
# plot_group_samples(data, n_no_strong=3, n_with_strong=3, seed=0, out_root=None)  # set out_root=OUT_ROOT to save

In [35]:
# Three datasets, each with 5% of exactly one strong anomaly type
# data_abs = generate_synthetic_dataset(seed=123, strong_rate=0.05, strong_kind="gauss_abs")
data_step = generate_synthetic_dataset(seed=123, strong_rate=0.05, strong_kind="step")
# data_em  = generate_synthetic_dataset(seed=123, strong_rate=0.05, strong_kind="gauss_em")

def strong_fraction(dataset):
    fracs = []
    for g in dataset["groups"]:
        fracs.append(np.mean(g["has_strong_anom"]))
    return float(np.mean(fracs))

# print("Strong fraction (gauss_abs dataset):", strong_fraction(data_abs))
print("Strong fraction (step dataset):     ", strong_fraction(data_step))
# print("Strong fraction (gauss_em dataset): ", strong_fraction(data_em))


Strong fraction (step dataset):      0.04


In [36]:
# plot_group_samples(data_abs, out_root="Images/SyntheticPlots_abs", n_no_strong=3, n_with_strong=3, seed=0)
plot_group_samples(data_step, out_root="Images/SyntheticPlots_step", n_no_strong=3, n_with_strong=3, seed=0)
# plot_group_samples(data_em, out_root="Images/SyntheticPlots_em", n_no_strong=3, n_with_strong=3, seed=0)


Saved Images/SyntheticPlots_step/synthetic_L100_W5_samples.png
Saved Images/SyntheticPlots_step/synthetic_L100_W10_samples.png
Saved Images/SyntheticPlots_step/synthetic_L200_W10_samples.png
Saved Images/SyntheticPlots_step/synthetic_L200_W20_samples.png
Saved Images/SyntheticPlots_step/synthetic_L500_W25_samples.png
Saved Images/SyntheticPlots_step/synthetic_L500_W50_samples.png


## 9) Export helper (optional)

In [ ]:
def save_dataset_npz(dataset: Dict[str, Any], out_path: str) -> None:
    out: Dict[str, np.ndarray] = {}
    for g in dataset["groups"]:
        gid = int(g["group_id"])
        prefix = f"g{gid}"
        out[f"{prefix}_length"] = np.array([g["length"]], dtype=np.int32)
        out[f"{prefix}_W"] = np.array([g["W"]], dtype=np.int32)
        out[f"{prefix}_spectra"] = g["spectra"].astype(np.float32)
        out[f"{prefix}_has_strong"] = g["has_strong_anom"].astype(np.int8)
        out[f"{prefix}_strong_labels"] = np.array(g["strong_labels"], dtype=object)
        out[f"{prefix}_weak_labels"] = np.array(g["weak_labels"], dtype=object)
        out[f"{prefix}_ignore_ranges"] = np.array(g["ignore_ranges"], dtype=object)

    ensure_dir(os.path.dirname(out_path) or ".")
    np.savez(out_path, **out)
    print(f"Wrote {out_path}")

# Example:
# save_dataset_npz(data, "Data/synthetic_spectra_param_playground.npz")